# Baseline Optimization Framework: Binary Cross-Entropy in Simple Linear Models
**Course Project: Class Imbalance in Machine Learning — Phase 1 Baseline**

### Project Overview
This notebook establishes the performance baseline for our class imbalance analysis by implementing a standard **Logistic Regression** classifier optimized via classic **Binary Cross-Entropy (BCE)** loss. This serves as the benchmark against which more complex handling methods—such as Focal Loss or algorithmic reweighting—are evaluated across 49 tabular datasets.

### Key Operational Targets:
1. **Mathematical Baseline:** Establish an empirical reference point using unweighted cross-entropy loss functions.
2. **First-Order Optimization:** Implement gradient descent via automatic differentiation (`autograd`) with an $L_2$ Tikhonov regularization penalty ($L_2$ regularization).
3. **Imbalance Characterization:** Catalog structural traits (imbalance ratios) across diverse data distributions while evaluating model performance using metrics resilient to class skew ($\text{ROC-AUC}$, $\text{PR-AUC}$, and $\text{F1-Score}$).

## 1. Global Dependencies & Environment Configuration
We load standard data processing libraries, statistical metric modules from `scikit-learn`, and the `autograd` functional array layer. Random seeds are locked globally to guarantee reproducibility across different runs.

In [ ]:

import logging
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import autograd.numpy as anp
from autograd import grad

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, average_precision_score, confusion_matrix,
)

np.random.seed(1000)
logging.basicConfig(level=logging.ERROR)

## 2. Mathematical Definition of Objective Functions
The classifier optimizes standard Binary Cross-Entropy (BCE). Unlike Focal Loss, this function treats all classification errors uniformly, making the loss gradient highly sensitive to dominant majority classes when severe class imbalance exists:

$$BCE(p) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

Numerical boundaries are applied via clipping to prevent optimization failure modes ($\log(0) \to -\infty$).

In [ ]:
#metrics

def binary_crossentropy(y, p):
    p = anp.clip(p, 1e-12, 1.0 - 1e-12)
    return -anp.mean(y * anp.log(p) + (1.0 - y) * anp.log(1.0 - p))

def accuracy(y, p):
    return np.mean(y == p)

## 3. Custom Object-Oriented Framework: Base Linear Interface
We define an abstract base container, `BaseEstimator`, which isolates basic matrix operations and prediction mappings. This ensures standard data type conversions remain consistent across training and inference.

In [ ]:
class BaseEstimator:
    def _setup_input(self, X, y=None):
        self.X = np.array(X, dtype=float)
        if y is not None:
            self.y = np.array(y, dtype=float)

    def predict(self, X):
        return self._predict(np.array(X, dtype=float))

## 4. Optimization Layer: Basic Parameterized Regression
The `BasicRegression` module implements an explicit, first-order gradient descent loop. It handles the feature matrix intercept modification, optimization termination conditions based on convergence tolerance ($\epsilon$), and parametric penalty calculations ($L_1$ lasso or $L_2$ ridge mechanics).

In [ ]:
class BasicRegression(BaseEstimator):
    def __init__(self, lr=0.001, penalty="None", C=0.01, tolerance=0.0001, max_iters=1000):
        self.C          = C
        self.penalty    = penalty
        self.tolerance  = tolerance
        self.lr         = lr
        self.max_iters  = max_iters
        self.errors     = []
        self.theta      = []
        self.n_samples  = None
        self.n_features = None
        self.cost_func  = None

    def _loss(self, w):
        raise NotImplementedError()

    def init_cost(self):
        raise NotImplementedError()

    def _add_penalty(self, loss, w):
        if self.penalty == "l1":
            loss += self.C * anp.abs(w[1:]).sum()
        elif self.penalty == "l2":
            loss += (0.5 * self.C) * (w[1:] ** 2).sum()
        return loss

    def _cost(self, X, y, theta):
        prediction = X.dot(theta)
        return self.cost_func(y, prediction)

    def fit(self, X, y=None):
        self._setup_input(X, y)
        self.init_cost()
        self.n_samples, self.n_features = self.X.shape
        self.theta = np.random.normal(size=(self.n_features + 1), scale=0.5)
        self.X     = self._add_intercept(self.X)
        self._train()

    @staticmethod
    def _add_intercept(X):
        b = np.ones([X.shape[0], 1])
        return np.concatenate([b, X], axis=1)

    def _train(self):
        self.theta, self.errors = self._gradient_descent()

    def _predict(self, X=None):
        X = self._add_intercept(X)
        return X.dot(self.theta)

    def _gradient_descent(self):
        theta  = self.theta.copy()
        errors = [self._cost(self.X, self.y, theta)]
        cost_d = grad(self._loss)
        for i in range(1, self.max_iters + 1):
            delta  = cost_d(theta)
            theta -= self.lr * delta
            errors.append(self._cost(self.X, self.y, theta))
            error_diff = np.linalg.norm(errors[i - 1] - errors[i])
            if error_diff < self.tolerance:
                break
        return theta, errors

## 5. Classifier Definition: Logistic Regression
We extend the optimization layer to build our unweighted `LogisticRegression` classifier. The parameters map directly through a stable sigmoid activation function ($\sigma(z) = \frac{1}{1 + e^{-z}}$) to produce proper probability boundaries.

In [ ]:

class LogisticRegression(BasicRegression):
    """Binary logistic regression with gradient descent optimizer."""

    def init_cost(self):
        self.cost_func = binary_crossentropy

    def _loss(self, w):
        loss = self.cost_func(self.y, self.sigmoid(anp.dot(self.X, w)))
        return self._add_penalty(loss, w)

    @staticmethod
    def sigmoid(x):
        return 0.5 * (anp.tanh(0.5 * x) + 1)

    def _predict(self, X=None):
        X = self._add_intercept(X)
        return self.sigmoid(X.dot(self.theta))

## 6. Pipeline Infrastructure & Mappings
We define data collection settings, target variable names, folder path locations, and verification routines.

In [ ]:
DATA_DIR = Path(r"C:\Users\Kasia\Downloads\class_imbalance (1)\class_imbalance")
N_SPLITS = 5
SEED     = 42

TARGET_MAP = {
    "dataset_1000_hypothyroid":           "binaryClass",
    "dataset_1002_ipums_la_98-small":      "binaryClass",
    "dataset_1004_synthetic_control":      "binaryClass",
    "dataset_1013_analcatdata_challenger": "Damaged",
    "dataset_1014_analcatdata_dmft":       "binaryClass",
    "dataset_1016_vowel":                  "binaryClass",
    "dataset_1018_ipums_la_99-small":      "binaryClass",
    "dataset_1020_mfeat-karhunen":         "binaryClass",
    "dataset_1021_page-blocks":            "binaryClass",
    "dataset_1022_mfeat-pixel":            "binaryClass",
    "dataset_1023_soybean":                "binaryClass",
    "dataset_1039_hiva_agnostic":          "label",
    "dataset_1045_kc1-top5":              "DL",
    "dataset_1049_pc4":                    "c",
    "dataset_1050_pc3":                    "c",
    "dataset_1056_mc1":                    "c",
    "dataset_1059_ar1":                    "defects",
    "dataset_1061_ar4":                    "defects",
    "dataset_1064_ar6":                    "defects",
    "dataset_1065_kc3":                    "c",
    "dataset_311_oil_spill":               "class",
    "dataset_312_scene":                   "Urban",
    "dataset_316_yeast_ml8":               "class14",
    "dataset_38_sick":                     "Class",
    "dataset_450_analcatdata_lawsuit":     "Laid.off",
    "dataset_463_backache":                "col_33",
    "dataset_757_meta":                    "binaryClass",
    "dataset_764_analcatdata_apnea3":      "binaryClass",
    "dataset_765_analcatdata_apnea2":      "binaryClass",
    "dataset_767_analcatdata_apnea1":      "binaryClass",
    "dataset_865_analcatdata_neavote":     "binaryClass",
    "dataset_867_visualizing_livestock":   "binaryClass",
    "dataset_875_analcatdata_chlamydia":   "binaryClass",
    "dataset_940_water-treatment":         "binaryClass",
    "dataset_947_arsenic-male-bladder":    "binaryClass",
    "dataset_949_arsenic-female-bladder":  "binaryClass",
    "dataset_950_arsenic-female-lung":     "binaryClass",
    "dataset_951_arsenic-male-lung":       "binaryClass",
    "dataset_954_spectrometer":            "binaryClass",
    "dataset_958_segment":                 "binaryClass",
    "dataset_962_mfeat-morphological":     "binaryClass",
    "dataset_966_analcatdata_halloffame":  "binaryClass",
    "dataset_968_analcatdata_birthday":    "binaryClass",
    "dataset_971_mfeat-fourier":           "binaryClass",
    "dataset_976_JapaneseVowels":          "binaryClass",
    "dataset_978_mfeat-factors":           "binaryClass",
    "dataset_980_optdigits":               "binaryClass",
    "dataset_984_analcatdata_draft":       "binaryClass",
    "dataset_987_collins":                 "binaryClass",
    "dataset_995_mfeat-zernike":           "binaryClass",
}

## 7. Operational Utility Functions
These helper functions manage categorical tokenization, isolate target variables, compute the dataset imbalance ratio ($IR = \frac{N_{majority}}{N_{minority}}$), and evaluate performance indicators across validation splits.

In [ ]:
def find_target(df, csv_path):
    target = TARGET_MAP.get(csv_path.stem)
    if target and target in df.columns:
        return target
    return df.columns[-1]


def encode_binary(series):
    vals = series.dropna().unique()
    if set(vals) <= {0, 1, 0.0, 1.0, True, False}:
        return series.astype(float)
    str_vals = set(str(v) for v in vals)
    if str_vals == {'-1', '1'}:
        return series.map({-1: 0, 1: 1, '-1': 0, '1': 1}).astype(float)
    mapping = {v: float(i) for i, v in enumerate(sorted(vals, key=str))}
    return series.map(mapping)


def imbalance_ratio(y):
    counts = np.bincount(y.astype(int))
    if len(counts) < 2 or counts.min() == 0:
        return float('inf')
    return round(counts.max() / counts.min(), 2)


def eval_fold(model, X_tr, y_tr, X_val, y_val):
    model.fit(X_tr, y_tr)
    proba = np.array(model._predict(X_val)).flatten()
    pred  = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    return {
        "roc_auc":      roc_auc_score(y_val, proba),
        "pr_auc":       average_precision_score(y_val, proba),
        "f1":           f1_score(y_val, pred, zero_division=0),
        "precision":    precision_score(y_val, pred, zero_division=0),
        "recall":       recall_score(y_val, pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    }

## 8. Baseline Execution Routine & Stratified Validation Loop
This execution engine loops through the target datasets, cleans missing records, imputes values via median strategy tracking, scales the feature metrics, and applies `StratifiedKFold` cross-validation.

In [ ]:
def main():
    results  = []
    datasets = sorted(DATA_DIR.glob("dataset_*.csv"))
    print(f"Znaleziono {len(datasets)} datasetow\n")

    for csv_path in tqdm(datasets, desc="Datasety"):
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[SKIP] {csv_path.stem} — blad odczytu: {e}")
            continue

        target_col = find_target(df, csv_path)

        if target_col not in df.columns:
            print(f"[SKIP] {csv_path.stem} — brak kolumny '{target_col}'")
            continue

        df = df.dropna(subset=[target_col])

        for col in df.columns:
            if col == target_col:
                continue
            try:
                df[col] = pd.to_numeric(df[col], errors='raise')
            except (ValueError, TypeError):
                df[col] = pd.Categorical(df[col]).codes.astype(float)
                df[col] = df[col].replace(-1, np.nan)

        try:
            y = encode_binary(df[target_col]).values.astype(float)
        except Exception as e:
            print(f"[SKIP] {csv_path.stem} — blad kodowania targetu: {e}")
            continue

        X = (df.drop(columns=[target_col])
               .fillna(df.drop(columns=[target_col]).median(numeric_only=True))
               .values.astype(float))

        if len(np.unique(y)) < 2:
            print(f"[SKIP] {csv_path.stem} — target ma tylko 1 klase")
            continue

        ir  = imbalance_ratio(y)
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        fold_metrics = []

        for fold_i, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
            X_tr,  X_val = X[tr_idx],  X[val_idx]
            y_tr,  y_val = y[tr_idx],  y[val_idx]

            scaler = StandardScaler()
            X_tr   = np.nan_to_num(scaler.fit_transform(X_tr), nan=0.0)
            X_val  = np.nan_to_num(scaler.transform(X_val),   nan=0.0)

            model = LogisticRegression(
                lr=0.01,
                max_iters=500,
                penalty="l2",
                C=0.01,
            )

            try:
                metrics = eval_fold(model, X_tr, y_tr, X_val, y_val)
                fold_metrics.append(metrics)
            except Exception as e:
                print(f"  [ERR]  {csv_path.stem} fold {fold_i}: {e}")

        if not fold_metrics:
            continue

        avg = {k: np.mean([m[k] for m in fold_metrics]) for k in fold_metrics[0]}
        std = {k: np.std( [m[k] for m in fold_metrics]) for k in fold_metrics[0]}

        results.append({
            "dataset":         csv_path.stem,
            "target_col":      target_col,
            "n_samples":       len(y),
            "n_features":      X.shape[1],
            "imbalance_ratio": ir,
            **{f"{k}_mean": round(avg[k], 4) for k in avg},
            **{f"{k}_std":  round(std[k], 4) for k in std},
        })

    if not results:
        print("\nnone results ")
        return

    results_df = pd.DataFrame(results)
    out_path   = DATA_DIR / "results_etap1.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\nZapisano → {out_path}")

    print("\n Outputs")
    print(results_df[["dataset", "imbalance_ratio", "roc_auc_mean",
                       "pr_auc_mean", "f1_mean", "recall_mean"]].to_string(index=False))

    print(f"\n── Srednie po wszystkich datasetach ──")
    for col in ["roc_auc_mean", "pr_auc_mean", "f1_mean", "recall_mean"]:
        print(f"  {col}: {results_df[col].mean():.4f}")

if __name__ == "__main__":
    main()